# P5 · Compteur de trafic

Avant de commencer : **Exécution → Modifier le type d'exécution → GPU T4**.

## 1. Récupérer le projet et installer les dépendances

Si le dossier existe déjà (cellule relancée), on récupère simplement la dernière version.

In [ ]:
import os
if not os.path.exists('/content/compteur-trafic-p5'):
    !git clone https://github.com/Atou-D10/compteur-trafic-p5.git /content/compteur-trafic-p5
%cd /content/compteur-trafic-p5
!git pull
!pip install -q -r requirements.txt

In [ ]:
import torch
print('GPU disponible :', torch.cuda.is_available())

## 2. Détection et suivi (tâche 1)

YOLO détecte les piétons, voitures, motos et bus sur chaque image, et ByteTrack donne à chaque objet un identifiant (`#id`) qui reste le même tant qu'il est visible. Les motards ne sont pas comptés comme piétons.

La vidéo `rue.mp4` (filmée par le groupe à Dakar, voir `data/README.md`) n'est pas dans Git : on la copie depuis Google Drive. Adaptez `CHEMIN_DRIVE` si elle est rangée ailleurs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHEMIN_DRIVE = '/content/drive/MyDrive/rue.mp4'
!cp "$CHEMIN_DRIVE" data/videos/rue.mp4
!ls -lh data/videos/

### 2.1 Quelques images avec boîtes et identifiants

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.detection import suivre_video, dessiner

images_voulues = [150, 600, 1200, 1800]   # numéros d'images à afficher (30 images = 1 s)
captures = {}
for numero, image, detections in suivre_video():
    if numero in images_voulues:
        captures[numero] = cv2.cvtColor(dessiner(image.copy(), detections), cv2.COLOR_BGR2RGB)
    if numero >= max(images_voulues):
        break

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, numero in zip(axes.flat, images_voulues):
    ax.imshow(captures[numero])
    ax.set_title(f'Image {numero} ({numero / 30:.0f} s)')
    ax.axis('off')
plt.tight_layout()
plt.show()

### 2.2 Vidéo annotée complète (environ 4 minutes de calcul)

In [ ]:
from src.detection import video_annotee
resume = video_annotee()

In [ ]:
!ffmpeg -y -loglevel error -i outputs/detection.mp4 -vcodec libx264 outputs/detection_web.mp4
from IPython.display import Video
Video('outputs/detection_web.mp4', embed=True, width=800)